LAB 10: Polynomial Fitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

This is the function for the helpers.

In [ ]:
def helpers(dataxs, datays, datasigs):
    S = np.sum(1/datasigs**2)
    Sx = np.sum(dataxs/datasigs**2)
    Sy = np.sum(datays/datasigs**2)
    Sxx = np.sum(dataxs**2/datasigs**2)
    Sxy = np.sum(dataxs*datays/datasigs**2)
    Del = S*Sxx - Sx**2
    return S, Sx, Sy, Sxx, Sxy, Del

This is the function for the vectors.

In [ ]:
def computecs(dataxs, datays, datasigs):
    S,Sx,Sy,Sxx,Sxy,Del = helpers(dataxs,datays,datasigs)
    cs = np.zeros(2);
    dcs = np.zeros(2);
    cs[0] = (Sxx*Sy - Sx*Sxy)/Del
    cs[1] = (S*Sxy - Sx*Sy)/Del
    dcs[0] = np.sqrt(Sxx/Del)
    dcs[1] = np.sqrt(S/Del)
    return cs, dcs

This is the function for the chi-squared statistic.

In [ ]:
def computechisq(dataxs, datays, datasigs, cs):
    chisq = np.sum((datays-cs[0]-cs[1]*dataxs)**2/datasigs**2)
    return chisq

Using the example data in the discussion:

In [ ]:
dataxs = np.linspace(0,1,6)
datays = np.array([3.085, 3.123, 3.224, 3.360, 3.438, 3.569])
datasigs = np.array([0.048, 0.053, 0.02, 0.005, 0.023, 0.07])

In [ ]:
def generatedata(n):
    data = np.zeros((3,n))
    data[0,:] = np.linspace(0, 1, n)
    data[1,:] = np.array([3.085, 3.123, 3.224, 3.360, 3.438, 3.569]) # generate sample data
    data[2,:] = np.array([0.048, 0.053, 0.02, 0.005, 0.023, 0.07]) # generate random values for uncertainty
    return data

In [ ]:
data = generatedata(6)
print(data)

We show the vector of parameters and the uncertainties.

In [ ]:
cs, dcs = computecs(dataxs, datays, datasigs)
print(cs)
print(dcs)

Then we compute for the chi-squared.

In [ ]:
chisq = computechisq(dataxs, datays, datasigs, cs)
print('Chi-squared:',chisq)
print('Degrees of Freedom:', dataxs.size - cs.size)
print(chisq/(dataxs.size - cs.size))

Then using the chi-squared, we plot the line of least fit.

We define a model for n = 2 (linear), n = 3 (quadratic), and n = 4 (quartic). 

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(2.9, 3.7)

ax.errorbar(dataxs, datays, yerr=datasigs, fmt='o', ecolor='blue', capsize=2, label='Data')
ax.plot(dataxs,cs[0]+cs[1]*dataxs,linestyle="--",color='black', label='Least-squares Fitting')

ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

PART A.

Now, we write a code for the polynomial fits of the given data above at n = 2, n = 3, n = 4.

In [ ]:
def backsub(U,bs):
    n = bs.size
    xs = np.zeros(n)
    for i in reversed(range(n)):
        xs[i] = (bs[i]-U[i,i+1:]@xs[i+1:])/U[i,i]
    return xs

In [ ]:
def gauelim(inA,inbs):
    A = np.copy(inA)
    bs = np.copy(inbs)
    n = bs.size

    for j in range(n-1):
        for i in range(j+1,n):
            coeff = A[i,j]/A[j,j]
            A[i,j:] -= coeff*A[j,j:]
            bs[i] -= coeff*bs[j]

    xs = backsub(A,bs)
    return xs

We define a function phi for n = 2 (quadratic), n = 3 (cubic), n = 4 (quartic).

In [ ]:
def phi(n,k,x):
    if n in (2,3,4):
        val = x**k
    return val

Function for normal equations

In [ ]:
def normalfit(data,phi,n):
    N = data.shape[1]
    A = np.zeros((N,n))
    for k in range(n):
        A[:,k] = phi(n,k,data[0,:])/data[2,:]
    bs= data[1,:]/data[2,:]
    cs = gauelim(A.T@A, A.T@bs)
    chisq = np.sum((bs - A@cs)**2)
    return cs, chisq

We plot the data.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(2.9, 3.7)


ax.errorbar(dataxs, datays, yerr=datasigs, fmt='o', ecolor='blue', capsize=2, label='Data')
ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

For n = 4:

In [ ]:
n=4
pcs4, pchisq4 = normalfit(data, phi, n)
print(pchisq4/(data.shape[1]-pcs4.size))

In [ ]:
xs4=np.linspace(0,1,6)
pfit4 = pcs4[0]+(pcs4[1]*xs4)+(pcs4[2]*xs4**2)+(pcs4[3]*xs4**3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(2.9, 3.7)


ax.errorbar(data[0,:], data[1,:], yerr=data[2,:], fmt='o', ecolor='blue', capsize=2, label='Data')
ax.plot(xs4,pfit4,linestyle="--",color='black', label=r'Polynomial $n=4$')
ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

For n = 3:

In [ ]:
pcs3, pchisq3 = normalfit(data, phi, 3)
print(pchisq3/(data.shape[1]-pcs3.size))

In [ ]:
xs3=np.linspace(0,1,6)
pfit3 = pcs3[0]+(pcs3[1]*xs3)+(pcs3[2]*xs3**2)
print(pcs3[0], pcs3[1], pcs3[2])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(2.9, 3.7)


ax.errorbar(data[0,:], data[1,:], yerr=data[2,:], fmt='o', ecolor='blue', capsize=2, label='Data')
ax.plot(xs3,pfit3,linestyle="--",color='black', label=r'Polynomial $n=3$')
ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

For n = 2:

In [ ]:
pcs2, pchisq2 = normalfit(data, phi, 2)
print(pchisq2/(data.shape[1]-pcs2.size))

In [ ]:
xs2=np.linspace(0,1,6)
pfit2 = pcs2[0]+(pcs2[1]*xs2)
print(pcs2[0], pcs2[1])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(2.9, 3.7)


ax.errorbar(data[0,:], data[1,:], yerr=data[2,:], fmt='o', ecolor='blue', capsize=2, label='Data')
ax.plot(xs2,pfit2,linestyle="--",color='black', label=r'Polynomial $n=2$')
ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

Combining the three graphs: We see that n=2 and n=3 seems to overlap with each other.

In [ ]:
#we see the similarity of the y-values of the line-of-best fit for n=2 and n=3
print(pfit3)
print(pfit2)

fig, ax = plt.subplots(figsize=(6, 4))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(2.9, 3.7)


ax.errorbar(data[0,:], data[1,:], yerr=data[2,:], fmt='o', ecolor='blue', capsize=2, label='Data')
ax.plot(xs2,pfit2,linestyle="--",color='black', label=r'Polynomial $n=2$')
ax.plot(xs3,pfit3,linestyle="--",color='red', label=r'Polynomial $n=3$')
ax.plot(xs4,pfit4,linestyle="--",color='green', label=r'Polynomial $n=4$')
ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

We define a function that solves for the difference of each fitted-line y-values p(x) and the datays we have. (Sorry sir di ko po alam paano ilagay to lahat sa isang function). Take note that the n here will depend on how many data/elements we have in our linspace for xs. Hence for instance in this example if we put n = 7, it will result in an error.

In [ ]:
#this is for n=4
def difference4(n):
    A = np.zeros(n) 
    for i in range(n):
        A[i] = datays[i]- pfit4[i] 
    print(A)
    return A

In [ ]:
#this is for n=3
def difference3(n):
    A = np.zeros(n) 
    for i in range(n):
        A[i] =  datays[i]- pfit3[i] 
    print(A)
    return A

In [ ]:
#this is for n=2
def difference2(n):
    A = np.zeros(n) 
    for i in range(n):
        A[i] = datays[i] - pfit2[i] 
    print(A)
    return A

In [ ]:
diff4 = difference4(6)
diff3 = difference3(6)
diff2 = difference2(6)
print(xs2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

#ax.errorbar(data[0,:], data[1,:], yerr=data[2,:], fmt='o', ecolor='blue', capsize=2, label='Data')
ax.axhline(0, color = 'black', linestyle = '--')
ax.plot(xs2,diff2,linestyle="--",color='black', label=r'Polynomial $n=2$')
ax.plot(xs3,diff3,linestyle="--",color='red', label=r'Polynomial $n=3$')
ax.plot(xs4,diff4,linestyle="--",color='green', label=r'Polynomial $n=4$')
ax.legend()
ax.set_xlabel(r'$x$')

plt.show()

PART B: 

We simply use the generate data function above and replaced the x and y-components into t and y (distance) respectively. Take note that since there is no inital velocity we transformed the quadratic function is now:$y = \frac{at^{2}}{2}$ 

In [ ]:
def generatedata(n):
    data = np.zeros((3,n))
    data[0,:] = np.array([6.87, 6.36, 5.80, 5.19, 4.50, 3.67, 2.60]) #this is the time component
    data[1,:] = np.array([12.5, 25.,37.5, 50., 62.5, 75., 87.5]) #this is the y-component
    data[2,:] = 1
    return data

In [ ]:
data = generatedata(7)
print(data)

We find the chi-square of the data. Since the equation is qudratic, our n must be equal to 3.

In [ ]:
pcs, pchisq = normalfit(data, phi, 3)
print(pchisq/(data.shape[1]-pcs.size))

In [ ]:
ts=np.array([6.87, 6.36, 5.80, 5.19, 4.50, 3.67, 2.60])
pfit = pcs[0]+(pcs[1]*ts)+(pcs[2]*ts**2)
print(pcs[0], pcs[1], pcs[2]) #the line of best fit has a constant c0 and c1 for x

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

Gravity = 2*pcs[2] #this is from the quadratic equation

ax.errorbar(data[0,:], data[1,:], yerr=data[2,:], fmt='o', ecolor='blue', capsize=2, label='Data')
ax.plot(ts,pfit,linestyle="--",color='black', label=r'$y = \frac{1}{2}gt^{2}$')
ax.text(0.05, .90, f'Gravity = {Gravity:.3f}', bbox=dict(facecolor='white', edgecolor='gray'))
ax.legend()
ax.set_xlabel(r'Time (in seconds)')
ax.set_ylabel(r'Height (in meters)')
plt.show()

Gravity is equal to the $2pcs[2]$ (this is from our quadratic equation). The gravity is approximately $-3.684\frac{m}{s^2}$. This value is close to the gravity of mercury or mars.